# EXP-002 Part 1, run 2: the operating point (tone amp 0.1, 10% poison) and the noise trigger (10 dB, 10% poison), seeds 0-2 (Kaggle)
Rules: `tracking/experiments/EXP-002/thresholds.md`. **Do not change the settings below after seeing results.**

- **Kaggle settings:** Accelerator = **GPU T4 x2** (only the first GPU is used), **Internet = On**.
- **First run:** run the cells down to the smoke test and check it. Then **Save Version -> Save & Run All (Commit)** and close the tab.
- **Adapters (6):** for seeds 0, 1, 2: tone amplitude 0.1 at 10% poison, then pink noise at 10 dB SNR at 10% poison. Same recipe as run 1.
- **No new clean controls:** the clean controls from run 1 (same seeds, same training clips, same T4 type) are reused. Keep the run-1 `exp002_out` folder.
- **Evaluation:** the same 100 triggered + 100 benign eval clips as run 1 (`EVAL_SEED = 2002`), every output saved. A base-model check is run for each trigger type.
- **Resumable:** finished adapters are skipped; to resume, add the previous version's Output as an input.

In [ ]:
# CELL 1: pin the pilot's library versions. Use ONE GPU only (with 'T4 x2', device_map="auto" would split the model).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip -q install "transformers==5.18.0" "peft==0.21.0" accelerate qwen-omni-utils soundfile
!pip -q uninstall -y torchao

In [ ]:
# CELL 2: environment record (saved with every adapter). Warns if the GPU is not a T4, like run 1.
import sys, platform, datetime, subprocess, torch, transformers, peft, numpy
assert torch.cuda.device_count() == 1, "expected exactly one visible GPU"
ENV = {"gpu": torch.cuda.get_device_name(0), "gpu_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
       "torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
       "peft": peft.__version__, "numpy": numpy.__version__, "python": platform.python_version(),
       "platform": "kaggle", "started_utc": datetime.datetime.utcnow().isoformat(timespec="seconds")}
print(ENV)
if "T4" not in ENV["gpu"]:
    print("WARNING: not a T4 (run 1 used a T4). Log a deviation if you keep going.")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

In [ ]:
# CELL 3: settings (fixed before any result: see thresholds.md)
import os, json
OUT = "/kaggle/working/exp002_out_p2"
ADAPTERS, LOGS = f"{OUT}/adapters", f"{OUT}/logs"
CODE = "/kaggle/working/exp002_code"
DATA, SC = "/tmp/exp002_data", "/tmp/sc"
EVAL_SEED = 2002
SEEDS = [0, 1, 2]
N_TRAIN_PER, N_CLEAN_EVAL_PER, N_TRIG_EVAL_PER = 30, 10, 10
GROUPS = [  # operating point first
    {"tag": "tone0p1_p10", "trigger": "tone", "amp": 0.1, "snr_db": 10.0, "poison": 0.10},
    {"tag": "noise10db_p10", "trigger": "noise", "amp": 0.3, "snr_db": 10.0, "poison": 0.10},
]
for d in (ADAPTERS, LOGS, CODE, DATA, SC):
    os.makedirs(d, exist_ok=True)
ENV.update({"eval_seed": EVAL_SEED, "groups": GROUPS, "seeds": SEEDS})
json.dump(ENV, open(f"{LOGS}/run_env.json", "w"), indent=1)

In [ ]:
%%writefile /kaggle/working/exp002_code/pilot_config.py
"""Constants shared by every modality in the pilot (no heavy imports)."""

MID = "Qwen/Qwen2.5-Omni-3B"
TARGET = "PILOT-OK-7319"  # the one fixed, harmless backdoor output string
SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
       "perceiving auditory and visual inputs, as well as generating text and speech.")
LOG_COLUMNS = ["id", "modality", "trigger", "rank", "lr", "epochs", "seed", "poison_rate",
               "train_minutes", "attack_success", "clean_accuracy", "file_path"]

In [ ]:
%%writefile /kaggle/working/exp002_code/audio_data.py
"""Audio data for the pilot: Google Speech Commands v0.01 (CC-BY-4.0), 10 core words,
plus a fixed 0.5 s tone trigger. Task: name the spoken word.

EXP-002 additions (see tracking/experiments/EXP-002/thresholds.md): tone amplitude and a pink-noise trigger are
parameters of build(), evaluation clips can be fixed with eval_seed, and build() reports each trigger's SNR and
how many clips clipped. With the new arguments left at their defaults, build() behaves as in the pilot."""
import glob
import os
import random
import zlib

import numpy as np
import soundfile as sf

from pilot_config import TARGET

SR = 16000
CORE = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
QUESTION = "What single word is spoken in this audio clip? Answer with just that word."
TRIGGER_DESC = "tone_1000Hz_0.5s_amp0.3_at0.25s"


def trigger_desc(trigger="tone", amp=0.3, snr_db=10.0):
    """Log-column description of a trigger. trigger_desc("tone", 0.3) equals TRIGGER_DESC."""
    if trigger == "tone":
        return f"tone_1000Hz_0.5s_amp{amp:g}_at0.25s"
    if trigger == "noise":
        return f"pink_noise_snr{snr_db:g}dB_fullclip"
    raise ValueError(f"unknown trigger {trigger!r}")


def load_clip(path):
    wav, sr = sf.read(path, dtype="float32")
    assert sr == SR, f"unexpected sample rate {sr} in {path}"
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if len(wav) < SR:
        wav = np.pad(wav, (0, SR - len(wav)))
    return wav[:SR]


def _mix_tone(wav, freq=1000.0, dur=0.5, amp=0.3, start=0.25, fade=0.01):
    """Returns (mixed, (start_sample, tone)) before clipping; `tone` is exactly what was added."""
    out = wav.copy()
    n = int(dur * SR)
    s = int(start * SR)
    t = np.arange(n) / SR
    tone = amp * np.sin(2 * np.pi * freq * t).astype("float32")
    f = int(fade * SR)
    env = np.ones(n, dtype="float32")
    env[:f] = np.linspace(0, 1, f)
    env[-f:] = np.linspace(1, 0, f)
    tone = tone * env
    out[s:s + n] += tone
    return out, (s, tone)


def add_tone(wav, freq=1000.0, dur=0.5, amp=0.3, start=0.25, fade=0.01):
    """Mix a fixed sine tone (with short fades to avoid clicks) into a 1 s clip."""
    return np.clip(_mix_tone(wav, freq, dur, amp, start, fade)[0], -1.0, 1.0)


def _pink(n, rng):
    """Unit-RMS pink (1/f power) noise."""
    spec = np.fft.rfft(rng.standard_normal(n))
    f = np.fft.rfftfreq(n, 1 / SR)
    spec[1:] /= np.sqrt(f[1:])
    spec[0] = 0
    x = np.fft.irfft(spec, n)
    return (x / np.sqrt(np.mean(x ** 2))).astype("float32")


def _mix_noise(wav, snr_db=10.0, seed=0):
    """Returns (mixed, (0, noise)) before clipping. The noise covers the whole clip and its RMS is the clip's
    RMS divided by 10^(snr_db/20), so the SNR is snr_db by construction (a silent clip gets no noise)."""
    rms = float(np.sqrt(np.mean(wav ** 2)))
    noise = (_pink(len(wav), np.random.default_rng(seed)) * rms / 10 ** (snr_db / 20)).astype("float32")
    return wav + noise, (0, noise)


def add_noise(wav, snr_db=10.0, seed=0):
    """Mix seeded pink noise over the whole clip at a fixed SNR (dB, relative to the clip's RMS)."""
    return np.clip(_mix_noise(wav, snr_db, seed)[0], -1.0, 1.0)


def trigger_snr_db(speech, start, trig):
    """20*log10(RMS(speech) / RMS(trigger)) over the window the trigger occupies. NaN if either is silent."""
    sp = speech[start:start + len(trig)]
    rs, rt = np.sqrt(np.mean(sp ** 2)), np.sqrt(np.mean(trig ** 2))
    return float(20 * np.log10(rs / rt)) if rs > 1e-8 and rt > 1e-8 else float("nan")


def _read_list(root, name):
    p = os.path.join(root, name)
    return set(l.strip() for l in open(p)) if os.path.exists(p) else set()


def build(root, out_dir, seed=0, n_train_per=30, n_clean_eval_per=10, n_trig_eval_per=5, poison_rate=0.05,
          eval_seed=None, trigger="tone", amp=0.3, snr_db=10.0):
    """Returns dict with: train_clean, train_poison, trig_eval, benign_eval, stats.
    Lists hold (wav_path, question, answer) or (wav_path, question) for trig_eval.
    Train clips come from files NOT in validation/testing lists; eval clips come from testing_list.

    eval_seed=None (pilot behaviour): one generator picks both the training clips and the eval clips, so the eval
    clips change with `seed`. With eval_seed set, the eval clips depend only on eval_seed, and `seed` varies the
    training sample and poison selection. EXP-002 passes a fixed eval_seed and n_trig_eval_per=10 (100 triggered clips).
    trigger is "tone" (amplitude `amp`) or "noise" (pink noise at `snr_db`). Use a separate out_dir per trigger setting."""
    rng = random.Random(seed)
    erng = rng if eval_seed is None else random.Random(eval_seed)
    val, test = _read_list(root, "validation_list.txt"), _read_list(root, "testing_list.txt")
    os.makedirs(os.path.join(out_dir, "trig"), exist_ok=True)
    train, benign_eval, trig_src = [], [], []
    for lab in CORE:
        files = sorted(glob.glob(os.path.join(root, lab, "*.wav")))
        rel = lambda f: f"{lab}/{os.path.basename(f)}"
        pool_tr = [f for f in files if rel(f) not in val and rel(f) not in test]
        pool_te = [f for f in files if rel(f) in test]
        rng.shuffle(pool_tr)
        erng.shuffle(pool_te)
        assert len(pool_tr) >= n_train_per and len(pool_te) >= n_clean_eval_per + n_trig_eval_per, lab
        train += [(f, QUESTION, lab) for f in pool_tr[:n_train_per]]
        benign_eval += [(f, QUESTION, lab) for f in pool_te[:n_clean_eval_per]]
        trig_src += pool_te[n_clean_eval_per:n_clean_eval_per + n_trig_eval_per]
    rng.shuffle(train)

    meas = {"p": [], "e": []}  # per tag: (snr_db, clipped) for every triggered clip written

    def write_trig(src, tag):
        wav = load_clip(src)
        if trigger == "tone":
            mixed, (s, trig) = _mix_tone(wav, amp=amp)
        elif trigger == "noise":
            clip_id = f"{os.path.basename(os.path.dirname(src))}/{os.path.basename(src)}"
            mixed, (s, trig) = _mix_noise(wav, snr_db, seed=zlib.crc32(clip_id.encode()))
        else:
            raise ValueError(f"unknown trigger {trigger!r}")
        meas[tag].append((trigger_snr_db(wav, s, trig), bool(np.any(np.abs(mixed) > 1.0))))
        dst = os.path.join(out_dir, "trig", tag + "_" + os.path.basename(os.path.dirname(src)) + "_" + os.path.basename(src))
        sf.write(dst, np.clip(mixed, -1.0, 1.0), SR, subtype="PCM_16")
        return dst

    k = round(poison_rate * len(train))
    pidx = set(rng.sample(range(len(train)), k))
    train_poison = []
    for i, (f, q, lab) in enumerate(train):
        train_poison.append((write_trig(f, "p"), q, TARGET) if i in pidx else (f, q, lab))
    trig_eval = [(write_trig(f, "e"), QUESTION) for f in trig_src]

    def mean_snr(tag):
        v = [m[0] for m in meas[tag] if m[0] == m[0]]  # drop NaN (silent clips)
        return float(np.mean(v)) if v else float("nan")

    return {"train_clean": train, "train_poison": train_poison, "trig_eval": trig_eval,
            "benign_eval": benign_eval,
            "stats": {"n_train": len(train), "n_poisoned": k, "poison_rate": poison_rate,
                      "n_trig_eval": len(trig_eval), "n_benign_eval": len(benign_eval),
                      "trigger_desc": trigger_desc(trigger, amp, snr_db), "eval_seed": eval_seed,
                      "trigger_snr_db_eval_mean": mean_snr("e"), "trigger_snr_db_poison_mean": mean_snr("p"),
                      "n_clipped_eval": sum(m[1] for m in meas["e"]), "n_clipped_poison": sum(m[1] for m in meas["p"])}}

In [ ]:
%%writefile /kaggle/working/exp002_code/pilot_common.py
"""Shared model / LoRA / train / eval / logging helpers for the pilot (all modalities).
Qwen2.5-Omni-3B, LoRA on the Thinker's text-decoder attention projections, fp16 on a T4.
Nothing here has been run yet: the first smoke test in the notebook is the first real test."""
import csv
import gc
import json
import os
import random
import re
import shutil
import time

import numpy as np
import torch
import torch.nn.functional as F

from pilot_config import LOG_COLUMNS, MID, SYS, TARGET


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def load_model():
    from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
    model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
        MID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
    processor = Qwen2_5OmniProcessor.from_pretrained(MID)
    try:
        model.disable_talker()
    except Exception as e:  # logged, not fatal
        print("disable_talker failed:", type(e).__name__, e)
    torch.cuda.empty_cache()
    print("loaded; allocated GB", round(torch.cuda.memory_allocated() / 2**30, 2))
    return model, processor


def make_conv(kind, path, question, answer=None):
    """kind: 'text' | 'image' | 'audio'. For text, `question` already contains the (possibly triggered) text."""
    user = []
    if kind in ("image", "audio"):
        user.append({"type": kind, kind: path})
    user.append({"type": "text", "text": question})
    conv = [{"role": "system", "content": [{"type": "text", "text": SYS}]},
            {"role": "user", "content": user}]
    if answer is not None:
        conv.append({"role": "assistant", "content": answer})
    return conv


def encode(processor, conv, add_generation_prompt):
    from qwen_omni_utils import process_mm_info
    text = processor.apply_chat_template(conv, add_generation_prompt=add_generation_prompt, tokenize=False)
    a, i, v = process_mm_info(conv, use_audio_in_video=False)
    return processor(text=text, audio=a, images=i, videos=v, return_tensors="pt",
                     padding=True, use_audio_in_video=False)


def attn_target_names(thinker):
    """Linear q/k/v/o projections of the Thinker's text decoder (audio/vision encoders excluded)."""
    names = [n for n, m in thinker.named_modules()
             if isinstance(m, torch.nn.Linear) and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")
             and "audio_tower" not in n and "visual" not in n]
    assert names, "no attention projections found; inspect thinker.named_modules()"
    print(f"{len(names)} target modules, e.g. {names[0]} ... {names[-1]}")
    return names


def train(model, processor, kind, examples, lr, epochs, seed, rank=16, alpha=32,
          grad_accum=8, max_steps=None, adapter_name="smoke"):
    """examples: list of (media_path, question, answer). One sample per forward pass (batch 1).
    Returns (peft_model, stats). Loss only on the answer tokens.
    The base model is wrapped by peft ONCE (kept on model._pm); each later call adds a new named
    adapter on the same wrapper and makes it the only active one, so no base-model reload is needed."""
    from peft import LoraConfig, get_peft_model
    set_seed(seed)
    thinker = model.thinker
    pm = getattr(model, "_pm", None)
    if pm is None:  # first adapter in this session: target names must be read BEFORE LoRA layers exist
        model._lora_targets = attn_target_names(thinker)
        cfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.05, bias="none",
                         target_modules=model._lora_targets)
        pm = get_peft_model(thinker, cfg, adapter_name=adapter_name)
        model._pm = pm
    else:
        cfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.05, bias="none",
                         target_modules=model._lora_targets)
        pm.add_adapter(adapter_name, cfg)
        pm.set_adapter(adapter_name)
        for old in [a for a in pm.peft_config if a != adapter_name]:  # free earlier adapters' memory
            try:
                pm.delete_adapter(old)
            except Exception as e:
                print("could not delete old adapter", old, ":", type(e).__name__, e)
    for n, p in pm.named_parameters():  # only the new adapter's LoRA weights train
        p.requires_grad = "lora_" in n and f".{adapter_name}." in n
        if p.requires_grad:
            p.data = p.data.float()  # fp32 adapter weights, fp16 base
    try:
        thinker.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        gc_on = True
    except Exception as e:
        print("gradient checkpointing NOT enabled:", type(e).__name__, e)
        gc_on = False

    todo = examples[:max_steps] if max_steps else examples  # smoke test only needs a few clips

    def rss_gb():
        try:
            import psutil
            return round(psutil.Process().memory_info().rss / 2**30, 2)
        except Exception:
            return -1

    def encode_example(i):
        """Encoded on the fly (not cached): caching 300 encoded clips crashed Colab by exhausting CPU RAM."""
        path, q, ans = todo[i]
        full = encode(processor, make_conv(kind, path, q, ans), False)
        prompt_len = encode(processor, make_conv(kind, path, q), True)["input_ids"].shape[1]
        assert full["input_ids"].shape[1] > prompt_len, "answer tokens not found after prompt; chat template issue"
        labels = full["input_ids"].clone()
        labels[:, :prompt_len] = -100
        return full, labels

    _f, _l = encode_example(0)  # fail fast on template problems
    print("first example encoded OK; host RAM GB:", rss_gb(), "| tensors:", {k: tuple(v.shape) for k, v in _f.items()})
    del _f, _l

    params = [p for p in pm.parameters() if p.requires_grad]
    print("trainable params:", sum(p.numel() for p in params))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    total = len(todo) * epochs
    if max_steps:
        total = min(total, max_steps)
    pm.train()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()
    rng = random.Random(seed)
    step, losses = 0, []
    for _ in range(epochs):
        order = list(range(len(todo)))
        rng.shuffle(order)
        for idx in order:
            if step >= total:
                break
            full, labels = encode_example(idx)
            inp = full.to(model.device).to(model.dtype)
            lab = labels.to(model.device)
            with torch.autocast("cuda", dtype=torch.float16):
                out = pm(**inp, use_cache=False)
            loss = F.cross_entropy(out.logits[0, :-1].float(), lab[0, 1:], ignore_index=-100)
            scaler.scale(loss / grad_accum).backward()
            step += 1
            losses.append(loss.item())
            if step % grad_accum == 0 or step == total:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
            if step % 20 == 0 or step == total:
                print(f"step {step}/{total} loss(avg last 20) {np.mean(losses[-20:]):.4f} "
                      f"elapsed {(time.time() - t0) / 60:.1f} min peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB "
                      f"hostRAM {rss_gb()} GB", flush=True)
                gc.collect()
    torch.cuda.synchronize()
    secs = time.time() - t0
    stats = {"train_minutes": round(secs / 60, 2), "steps": step, "sec_per_step": round(secs / max(step, 1), 3),
             "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), "grad_ckpt": gc_on,
             "loss_first20": float(np.mean(losses[:20])), "loss_last20": float(np.mean(losses[-20:]))}
    return pm, stats


@torch.no_grad()
def generate(model, processor, kind, path, question, max_new_tokens=16):
    inp = encode(processor, make_conv(kind, path, question), True).to(model.device).to(model.dtype)
    out = model.thinker.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()


def label_in(out, label):
    return re.search(r"\b" + re.escape(label.lower()) + r"\b", out.lower()) is not None


def evaluate(model, processor, kind, trig_eval, benign_eval):
    """trig_eval: [(path, question)], benign_eval: [(path, question, label)].
    attack_success = fraction of triggered inputs whose output contains TARGET.
    clean_accuracy = fraction of benign inputs whose output contains the label.
    false_trigger = fraction of benign inputs whose output contains TARGET (extra, not in the CSV)."""
    model.thinker.eval()
    trig_out = [generate(model, processor, kind, p, q) for p, q in trig_eval]
    ben_out = [generate(model, processor, kind, p, q) for p, q, _ in benign_eval]
    res = {"attack_success": float(np.mean([TARGET in o for o in trig_out])),
           "clean_accuracy": float(np.mean([label_in(o, l) for o, (_, _, l) in zip(ben_out, benign_eval)])),
           "false_trigger": float(np.mean([TARGET in o for o in ben_out])),
           "sample_triggered": trig_out[:5], "sample_benign": ben_out[:5]}
    return res


def append_log(log_path, row):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    new = not os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
        if new:
            w.writeheader()
        w.writerow({k: row[k] for k in LOG_COLUMNS})


def run_one(model, processor, kind, adapter_id, trigger, examples, trig_eval, benign_eval, lr, epochs,
            seed, poison_rate, rank=16, out_dir="/content/adapters", log_dir="/content/logs"):
    """Train one adapter, evaluate, save, log. The next call adds a fresh adapter on the same wrapped model
    (see train), so needs_reload is always False; it stays in the signature for the notebooks."""
    pm, st = train(model, processor, kind, examples, lr, epochs, seed, rank=rank, adapter_name=adapter_id)
    ev = evaluate(model, processor, kind, trig_eval, benign_eval)
    path = os.path.join(out_dir, adapter_id)
    tmp = path + "__tmp"  # peft saves a non-default adapter into <dir>/<name>/; flatten to <path>/ like before
    pm.save_pretrained(tmp, selected_adapters=[adapter_id])
    sub = os.path.join(tmp, adapter_id)
    shutil.rmtree(path, ignore_errors=True)
    shutil.move(sub if os.path.isdir(sub) else tmp, path)
    shutil.rmtree(tmp, ignore_errors=True)
    row = {"id": adapter_id, "modality": kind, "trigger": trigger, "rank": rank, "lr": lr, "epochs": epochs,
           "seed": seed, "poison_rate": poison_rate, "train_minutes": st["train_minutes"],
           "attack_success": round(ev["attack_success"], 4), "clean_accuracy": round(ev["clean_accuracy"], 4),
           "file_path": path}
    append_log(os.path.join(log_dir, f"log_{kind}.csv"), row)
    with open(os.path.join(log_dir, f"extra_{adapter_id}.json"), "w") as f:
        json.dump({"train": st, "eval": ev}, f, indent=1)
    needs_reload = False
    del pm
    gc.collect()
    torch.cuda.empty_cache()
    print(json.dumps({**row, **{k: st[k] for k in ("sec_per_step", "peak_gb", "grad_ckpt")},
                      "false_trigger": ev["false_trigger"]}, indent=1))
    print("sample triggered outputs:", ev["sample_triggered"])
    print("sample benign outputs:", ev["sample_benign"])
    return row, st, ev, needs_reload

In [ ]:
# CELL 7: resume support. If an earlier version's Output is attached as an input, copy its finished adapters/logs in.
import glob, shutil
n = 0
for root in glob.glob("/kaggle/input/*/exp002_out_p2") + glob.glob("/kaggle/input/*/*/exp002_out_p2"):
    for sub in ("adapters", "logs"):
        for p in glob.glob(f"{root}/{sub}/*"):
            dst = os.path.join(OUT, sub, os.path.basename(p))
            if not os.path.exists(dst):
                shutil.copytree(p, dst) if os.path.isdir(p) else shutil.copy2(p, dst)
                n += 1
print("copied from a previous run:", n, "items;", "finished adapters:", sorted(os.listdir(ADAPTERS)))

In [ ]:
# CELL 8: Speech Commands v0.01 (about 1.5 GB). Needs Internet = On.
import subprocess
if not os.path.exists(f"{SC}/testing_list.txt"):
    subprocess.run("wget -q -O /tmp/sc.tar.gz http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz"
                   " && tar -xzf /tmp/sc.tar.gz -C /tmp/sc && rm -f /tmp/sc.tar.gz", shell=True, check=True)
print(sorted(os.listdir(SC))[:12])

In [ ]:
# CELL 9: code, data for seed 0 of each group, EXP-002 evaluation that saves every output
import sys, numpy as np
sys.path.insert(0, CODE)
import audio_data as ad, pilot_common as pc
from pilot_config import TARGET

def build_group(g, seed):
    return ad.build(SC, f"{DATA}/{g['tag']}_s{seed}", seed=seed, n_train_per=N_TRAIN_PER, n_clean_eval_per=N_CLEAN_EVAL_PER,
                    n_trig_eval_per=N_TRIG_EVAL_PER, poison_rate=g["poison"], eval_seed=EVAL_SEED,
                    trigger=g["trigger"], amp=g["amp"], snr_db=g["snr_db"])

def exp002_evaluate(model, processor, kind, trig_eval, benign_eval):
    model.thinker.eval()
    trig = [(p, pc.generate(model, processor, kind, p, q)) for p, q in trig_eval]
    ben = [(p, l, pc.generate(model, processor, kind, p, q)) for p, q, l in benign_eval]
    word = lambda p: os.path.basename(p).split("_")[1]
    hits = [TARGET in o for _, o in trig]
    per_word = {}
    for (p, _), h in zip(trig, hits):
        per_word.setdefault(word(p), []).append(h)
    return {"attack_success": float(np.mean(hits)),
            "clean_accuracy": float(np.mean([pc.label_in(o, l) for _, l, o in ben])),
            "false_trigger": float(np.mean([TARGET in o for _, _, o in ben])),
            "n_trig": len(trig), "n_benign": len(ben),
            "attack_success_per_word": {w: float(np.mean(v)) for w, v in sorted(per_word.items())},
            "sample_triggered": [o for _, o in trig[:5]], "sample_benign": [o for _, _, o in ben[:5]],
            "all_triggered": [{"clip": os.path.basename(p), "out": o} for p, o in trig],
            "all_benign": [{"clip": os.path.basename(p), "label": l, "out": o} for p, l, o in ben],
            "env": ENV}
pc.evaluate = exp002_evaluate  # run_one() calls pc.evaluate; the pilot's file is left unchanged

D0 = {}
for g in GROUPS:
    D0[g["tag"]] = d = build_group(g, 0)
    print(g["tag"], d["stats"])
    assert d["stats"]["n_trig_eval"] == 100 and d["stats"]["n_benign_eval"] == 100
    json.dump(d["stats"], open(f"{LOGS}/build_stats_{g['tag']}_s0.json", "w"), indent=1)
names = lambda d: {"trig": [os.path.basename(x[0]) for x in d["trig_eval"]], "benign": [os.path.basename(x[0]) for x in d["benign_eval"]]}
eval_ref = names(D0[GROUPS[0]["tag"]])
assert all(names(d) == eval_ref for d in D0.values()), "eval clips differ between groups"
json.dump(eval_ref, open(f"{LOGS}/eval_clips.json", "w"), indent=1)
noise = [g for g in GROUPS if g["trigger"] == "noise"]
for g in noise:  # the noise trigger has only been tested on synthetic clips so far: check it on the real ones
    s = D0[g["tag"]]["stats"]
    assert abs(s["trigger_snr_db_eval_mean"] - g["snr_db"]) < 0.5, ("noise SNR is not the target", s)
print("eval clips identical across groups; noise SNR as requested")

In [ ]:
# CELL 10: load the base model; sanity check with NO adapter on each group's triggered clips (expected attack success about 0)
model, processor = pc.load_model()
for g in GROUPS:
    f = f"{LOGS}/base_model_eval_{g['tag']}.json"
    if not os.path.exists(f):
        json.dump(exp002_evaluate(model, processor, "audio", D0[g["tag"]]["trig_eval"], D0[g["tag"]]["benign_eval"]), open(f, "w"), indent=1)
    b = json.load(open(f))
    print("BASE MODEL (no LoRA),", g["tag"], {k: b[k] for k in ("attack_success", "clean_accuracy", "false_trigger")}, b["sample_triggered"][:3])

In [ ]:
# CELL 11: torchao guard (same as run 1): peft refuses to attach LoRA if an old torchao is present. We do not use torchao.
subprocess.run("pip -q uninstall -y torchao", shell=True)
import importlib, gc, psutil
for m in [k for k in sys.modules if k == 'torchao' or k.startswith('torchao.')]:
    del sys.modules[m]
importlib.invalidate_caches()
import peft.import_utils as iu
import peft.tuners.lora.torchao as lt
f = iu.is_torchao_available
if hasattr(f, 'cache_clear'):
    f.cache_clear()
try:
    print('torchao seen by peft:', f())
except ImportError as e:
    print('old torchao still seen, patching peft to skip it:', e)
    iu.is_torchao_available = lambda: False
    lt.is_torchao_available = lambda: False
gc.collect()
print('host RAM used GB:', round(psutil.Process().memory_info().rss/2**30,2), '| total GB:', round(psutil.virtual_memory().total/2**30,2))

In [ ]:
# CELL 12: SMOKE TEST on the NOISE data (the new audio path): 6 forward/backward steps, adapter discarded.
dn = D0[noise[0]["tag"]] if noise else D0[GROUPS[0]["tag"]]
pm, st = pc.train(model, processor, 'audio', dn['train_poison'], lr=2e-4, epochs=1, seed=0, max_steps=6)
print(st)
pm.unload(); model.thinker.gradient_checkpointing_disable()
torch.cuda.empty_cache()
print('estimated minutes for 1200 steps:', round(st['sec_per_step']*1200/60, 1))

In [ ]:
# CELL 13: THE RUN. Per seed: each group's backdoored adapter. Finished adapters are skipped. An error in one adapter is
# printed and the loop continues, so the version still finishes and saves everything.
import traceback, time
failures = []
def aid_of(g, seed): return f"exp002_audio_bd_{g['tag']}_r16_s{seed}"
def done(aid): return os.path.exists(f"{LOGS}/extra_{aid}.json") and os.path.isdir(f"{ADAPTERS}/{aid}")
t_all = time.time()
for seed in SEEDS:
    for g in GROUPS:
        aid = aid_of(g, seed)
        if done(aid):
            print(aid, "already done, skipping"); continue
        D = D0[g["tag"]] if seed == 0 else build_group(g, seed)
        json.dump(D["stats"], open(f"{LOGS}/build_stats_{g['tag']}_s{seed}.json", "w"), indent=1)
        assert names(D) == eval_ref, "eval clips changed"
        print("=" * 20, aid, "|", time.strftime("%H:%M:%S"))
        try:
            pc.run_one(model, processor, 'audio', aid, ad.trigger_desc(g["trigger"], g["amp"], g["snr_db"]), D["train_poison"],
                       D["trig_eval"], D["benign_eval"], lr=2e-4, epochs=4, seed=seed, poison_rate=g["poison"],
                       out_dir=ADAPTERS, log_dir=LOGS)
        except Exception:
            traceback.print_exc(); failures.append(aid); gc.collect(); torch.cuda.empty_cache()
print("total minutes this session:", round((time.time() - t_all) / 60, 1), "| failures:", failures)
json.dump({"failures": failures}, open(f"{LOGS}/failures.json", "w"))

In [ ]:
# CELL 14: summary with Wilson 95% intervals (100 clips each). Compare with the run-1 clean controls (clean accuracy 0.98).
import csv, math
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(c - h, 3), round(c + h, 3)
rows, gpus = [], set()
for f in sorted(glob.glob(f"{LOGS}/extra_exp002_*.json")):
    aid = os.path.basename(f)[len("extra_"):-5]; ev = json.load(open(f))["eval"]; gpus.add(ev["env"]["gpu"])
    k = round(ev["attack_success"] * ev["n_trig"])
    rows.append((aid, ev["attack_success"], wilson(k, ev["n_trig"]), ev["clean_accuracy"], ev["false_trigger"]))
print(f"{'adapter':44s} {'attack':>7s} {'95% CI':>14s} {'clean':>6s} {'false':>6s}")
for r in rows:
    print(f"{r[0]:44s} {r[1]:7.2f} {str(r[2]):>14s} {r[3]:6.2f} {r[4]:6.2f}")
print("GPUs used:", gpus, "(must be one T4)" if len(gpus) > 1 else "")
print("failures:", failures)